<a href="https://colab.research.google.com/github/UlaStats/MSc-project-pipe-failure-prediction/blob/main/Modelling_RNN_and_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch.nn as nn
from google.colab import drive
import pandas as pd
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
import numpy as np

In [54]:
drive.mount("/content/drive", force_remount = True)

Mounted at /content/drive


In [108]:
X_train_sequential = pd.read_csv("/content/drive/MyDrive/MSc project/X_train_sequential.csv", encoding = "latin1")
y_train_sequential = pd.read_csv("/content/drive/MyDrive/MSc project/y_train_sequential.csv", encoding = "latin1")

X_valid_sequential = pd.read_csv("/content/drive/MyDrive/MSc project/X_valid_sequential.csv", encoding = "latin1")
y_valid_sequential = pd.read_csv("/content/drive/MyDrive/MSc project/X_valid_sequential.csv", encoding = "latin1")

X_test_sequential = pd.read_csv("/content/drive/MyDrive/MSc project/X_test_sequential.csv", encoding = "latin1")
y_test_sequential = pd.read_csv("/content/drive/MyDrive/MSc project/y_test_sequential.csv", encoding = "latin1")


In [109]:
# convert categorical columns

X_train_sequential['Lining'] = X_train_sequential['Lining'].astype("category")
X_train_sequential['Surface'] = X_train_sequential['Surface'].astype("category")
X_train_sequential['Soil'] = X_train_sequential['Soil'].astype("category")
X_train_sequential['Material'] = X_train_sequential['Material'].astype("category")


X_valid_sequential['Lining'] = X_valid_sequential['Lining'].astype("category")
X_valid_sequential['Surface'] = X_valid_sequential['Surface'].astype("category")
X_valid_sequential['Soil'] = X_valid_sequential['Soil'].astype("category")
X_valid_sequential['Material'] = X_valid_sequential['Material'].astype("category")


X_test_sequential['Lining'] = X_test_sequential['Lining'].astype("category")
X_test_sequential['Surface'] = X_test_sequential['Surface'].astype("category")
X_test_sequential['Soil'] = X_test_sequential['Soil'].astype("category")
X_test_sequential['Material'] = X_test_sequential['Material'].astype("category")


In [110]:
# convert to tidy data

X_train_sequential_tidy = pd.get_dummies(X_train_sequential)
X_test_sequential_tidy = pd.get_dummies(X_test_sequential)


In [111]:
# normalise data

scaler = StandardScaler()

for feature in X_train_sequential_tidy[["Diameter", "Length", "Soil_pH", "Frost_days", "Hydrogen Ion", "Free chlorine", "Age", "Previous bursts"]]:
  X_train_sequential_tidy[feature] = scaler.fit_transform(X_train_sequential_tidy[[feature]])



In [112]:
# normalise data

scaler = StandardScaler()

for feature in X_test_sequential_tidy[["Diameter", "Length", "Soil_pH", "Frost_days", "Hydrogen Ion", "Free chlorine", "Age", "Previous bursts"]]:
  X_test_sequential_tidy[feature] = scaler.fit_transform(X_test_sequential_tidy[[feature]])

In [113]:
# filter response vector to contain only result for latest burst

index = X_train_sequential_tidy.groupby("Asset.ID")["Age"].nlargest(1).reset_index(level = 0, drop = True).index
y_train_sequential_tidy = y_train_sequential.loc[index]

index = X_test_sequential_tidy.groupby("Asset.ID")["Age"].nlargest(1).reset_index(level = 0, drop = True).index
y_test_sequential_tidy = y_test_sequential.loc[index]

In [114]:
# remove asset ID

X_train_sequential_tidy = X_train_sequential_tidy.drop("Asset.ID", axis = 1)
X_test_sequential_tidy = X_test_sequential_tidy.drop("Asset.ID", axis = 1)

In [115]:
# add empty columns to test data so that the number of columns is equal for both (requirement for model testing)

X_test_sequential_tidy["Material_Steel"] = 0
X_test_sequential_tidy["Material_Steel"] = X_test_sequential_tidy["Material_Steel"].astype(bool)

X_test_sequential_tidy["Lining_PRESENT"] = 0
X_test_sequential_tidy["Lining_PRESENT"] = X_test_sequential_tidy["Lining_PRESENT"].astype(bool)

In [116]:
# convert data frame to numpy array
X_train_numpy = X_train_sequential_tidy.to_numpy(dtype = np.float32)
X_test_numpy = X_test_sequential_tidy.to_numpy(dtype = np.float32)

In [117]:
# convert numpy array to tensor

X_train_tensor = tf.convert_to_tensor(X_train_numpy)

X_test_tensor = tf.convert_to_tensor(X_test_numpy)

In [118]:
# adjust tensor shape

X_train_tensor = tf.reshape(X_train_tensor, (720, 5, 28))

X_test_tensor = tf.reshape(X_test_tensor, (44, 5, 28))

# Modelling

In [76]:
from keras import models
from keras import layers


##### RNN

In [209]:
model = models.Sequential([
    layers.SimpleRNN(
        units=8,
        input_shape=(5, 28),
        activation='tanh',
        return_sequences=False
    ),
    layers.Dense(1, activation='relu')
])

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae', "root_mean_squared_error", "r2_score"]
)

model.summary()

# Train
history = model.fit(
    X_train_tensor,
    y_train_sequential_tidy,
    epochs=100,
    batch_size=16
)




Model: "sequential_45"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_44 (SimpleRNN)       │ (None, 8)              │           296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_45 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 305 (1.19 KB)

 Trainable params: 305 (1.19 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 12.4429 - mae: 2.0871 - r2_score: -0.0655 - root_mean_squared_error: 3.5275
Epoch 2/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 12.2796 - mae: 2.0739 - r2_score: -0.0515 - root_mean_squared_error: 3.5042
Epoch 3/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 12.1217 - mae: 2.0560 - r2_score: -0.0380 - root_mean_squared_error: 3.4816
Epoch 4/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 11.9322 - mae: 2.0359 - r2_score: -0.0218 - root_mean_squared_error: 3.4543
Epoch 5/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 11.7412 - mae: 2.0212 - r2_score: -0.0054 - root_mean_squared_error: 3.4265
Epoch 6/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 11.4285 - mae: 2.0052 - r2_score: 0.0214 - root_mean_squared_error: 3.3806 
Epoch 7/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 10.8782 - mae: 1.9956 - r2_score: 0.0685 - root_mean_squared_error: 3.2982
Epoch 8/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - los

KeyboardInterrupt: 

In [207]:
model.evaluate(X_test_tensor, y_test_sequential_tidy)

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 1.0944 - mae: 0.6892 - r2_score: -6.9475 - root_mean_squared_error: 1.0461 


[1.0943641662597656, 0.6891942620277405, 1.0461186170578003, -6.94750452041626]

##### LSTM

In [217]:
model_LSTM = models.Sequential([
    layers.LSTM(units = 8,
                input_shape=(5, 28)),
    layers.Dense(1, activation = "relu")
])


model_LSTM.compile(
    optimizer='adam',
    loss='mean_squared_error',
    metrics=['mae']
)


model_LSTM.fit(
    X_train_tensor,
    y_train_sequential_tidy,
    epochs=100,
    batch_size=16
)


Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 12.3061 - mae: 2.0507
Epoch 2/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 11.8108 - mae: 2.0076
Epoch 3/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 11.2953 - mae: 1.9771
Epoch 4/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 10.7894 - mae: 1.9603
Epoch 5/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 10.4304 - mae: 1.9831
Epoch 6/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 10.2525 - mae: 2.0003
Epoch 7/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 10.1636 - mae: 2.0093
Epoch 8/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 10.1082 - mae: 2.0104
Epoch 9/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 10.0728 - mae: 1.9984
Epoch 10/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 10.0303 - mae: 1.9998
Epoch 11/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 10.0049 - mae: 1.9995
Epoch 12/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 9.9796 - mae: 1.9829 
Epoch 13/100
45/45 ━━━━━━━━━━━━━━━━━━

In [218]:
model_LSTM.evaluate(X_test_tensor, y_test_sequential_tidy)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 6.9446 - mae: 2.0281 


[6.944572925567627, 2.0281355381011963]

In [ ]:
 1.8548